# 03. Нейросетевая модель

По результатам ноутбука 02 установлено, что критерии точности и выполнения
физических ограничений дают противоположные ответы о способе применения
факторизации. Здесь проверяется архитектура, снимающая это противоречие:

$$R_{pred} = NN(x)\cdot\bigl[1 - e^{-\varphi}\bigr],\qquad
x = (1000/T,\; \theta_2/T_{12},\; \theta_3/T_3)$$

Множитель входит в модель структурно, а функция потерь вычисляется по $R$ —
деления на малую величину нет нигде.

Сравниваются четыре модели на четырёх разбиениях по трём группам метрик.

In [ ]:
%load_ext autoreload
%autoreload 2

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from co2relax.data import load_core, TARGETS, FIG_DIR, REPORT_DIR
from co2relax.splits import SPLITS
from co2relax.train import Config, train_model, predict, jacobian
from co2relax.evaluate import compare_all, timing
from co2relax.models import device

pd.set_option("display.width", 250)
pd.set_option("display.float_format", "{:.3e}".format)
FIG_DIR.mkdir(parents=True, exist_ok=True)

core = load_core()
print("устройство:", device(), "| точек:", len(core))

## 1. Полное сравнение

Четыре модели:

| Модель | Устройство |
|---|---|
| сеть с физикой | $R = NN(x)\cdot[1-e^{-\varphi}]$, потери по $R$ |
| сеть без физики | та же сеть, предсказывает $R$ напрямую — изолирует вклад структуры |
| сплайн по R | кубическая интерполяция релаксационного члена |
| сплайн по A | интерполяция множителя, $R$ восстанавливается умножением |

In [ ]:
cfgs = {
    "сеть с физикой":  Config(model="physics", hidden=128, depth=4, epochs=6000),
    "сеть без физики": Config(model="plain",   hidden=128, depth=4, epochs=6000),
}

t0 = time.time()
res = compare_all(core, SPLITS, cfgs, seed=0)
print(f"{time.time()-t0:.0f} с")

res.to_csv(REPORT_DIR / "03_comparison.csv", index=False)
res.head()

## 2. Точность

Медианная относительная ошибка по восстановленному $R$. Точки вблизи нулевой
поверхности исключены из относительной метрики (там деление на малую величину
даёт произвольные значения независимо от качества модели).

In [ ]:
acc = res.pivot_table(index=["разбиение", "канал"], columns="модель", values="rel_median")
acc = acc[["сеть с физикой", "сеть без физики", "сплайн по R", "сплайн по A"]]
acc

In [ ]:
# 95-й процентиль — поведение в неблагоприятных случаях
res.pivot_table(index=["разбиение", "канал"], columns="модель", values="rel_p95")[
    ["сеть с физикой", "сеть без физики", "сплайн по R", "сплайн по A"]]

### Вклад физической структуры

Отношение ошибок контрольной модели к физически-информированной при одинаковой
архитектуре. Изолирует эффект факторизации от эффекта самого перехода
к нейросетям.

In [ ]:
p = res.pivot_table(index=["разбиение", "канал"], columns="модель", values="rel_median")
gain = (p["сеть без физики"] / p["сеть с физикой"]).round(2).rename("выигрыш, раз")
gain.to_frame()

## 3. Физические ограничения

Две проверки. Обе агрегируются по максимуму: ограничение либо выполняется
везде, либо нарушено.

**Невязка на равновесии** — значение $|R|$ в точках $T = T_{12} = T_3$,
где истина известна аналитически и равна нулю, нормированное на характерный
масштаб $R$.

**Доля неверного знака** — часть тестовых точек, где модель ошиблась
направлением процесса.

In [ ]:
eqv = res.pivot_table(index=["разбиение", "канал"], columns="модель", values="равновесие")
eqv[["сеть с физикой", "сеть без физики", "сплайн по R", "сплайн по A"]]

In [ ]:
sign = res.pivot_table(index="модель", values="доля_неверный_знак", aggfunc=["max", "mean"])
sign.columns = ["максимум", "в среднем"]
sign.sort_values("максимум")

Машинный ноль ($\sim10^{-16}$) у моделей с факторизацией означает, что
ограничение выполняется тождественно, а не аппроксимируется: при $\varphi = 0$
множитель обращается в ноль при любых весах сети.

## 4. Физическая интерпретация невязки

Оценка скорости паразитного дрейфа равновесного состояния.

In [ ]:
n = 1e23                      # м^-3, характерная плотность
theta2 = 960.0
T12_ref = 2000.0

R_scale = np.median(np.abs(core["R_VV23_12"][core["R_VV23_12"] != 0])) * n**2
i_mean = 1.0 / (np.exp(theta2 / T12_ref) - 1)      # среднее число квантов моды 12
W12 = n * i_mean

rows = []
for m in eqv.columns:
    v = eqv[m].max()
    if v <= 0 or not np.isfinite(v):
        rows.append(dict(модель=m, невязка=v, время_дрейфа_с=np.inf))
        continue
    rows.append(dict(модель=m, невязка=v, время_дрейфа_с=W12 / (v * R_scale)))

drift = pd.DataFrame(rows).sort_values("время_дрейфа_с", ascending=False)
print(f"R_scale = {R_scale:.2e} м^-3 с^-1,  W12 = {W12:.2e} м^-3")
print("газодинамическое время ~ 1e-3 с\n")
drift

Время дрейфа, сопоставимое с газодинамическим или меньшее, означает, что
равновесное состояние будет самопроизвольно эволюционировать быстрее, чем
развивается течение.

## 5. Кривые обучения

In [ ]:
tr, te = SPLITS["прореживание сетки"](core)

fig, ax = plt.subplots(figsize=(7, 4))
for name, cfg in cfgs.items():
    _, hist = train_model(tr, cfg)
    ax.semilogy(hist[:, 0], label=f"{name}, обучение", lw=1)
    ax.semilogy(hist[:, 1], "--", label=f"{name}, валидация", lw=1)
ax.set_xlabel("эпоха"); ax.set_ylabel("функция потерь")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.savefig(FIG_DIR / "03_learning_curves.png", dpi=150, bbox_inches="tight")

## 6. Предсказание против истины

In [ ]:
net, _ = train_model(tr, cfgs["сеть с физикой"])
pred = predict(net, te)

fig, axs = plt.subplots(1, 3, figsize=(13, 4))
R0 = np.median(np.abs(core[TARGETS[1]][core[TARGETS[1]] != 0]))
for ax, t, j in zip(axs, TARGETS, range(3)):
    true = te[t].to_numpy()
    ax.plot(np.arcsinh(true / R0), np.arcsinh(pred[:, j] / R0), ".", ms=3)
    lim = [np.arcsinh(true / R0).min(), np.arcsinh(true / R0).max()]
    ax.plot(lim, lim, "k-", lw=0.7)
    ax.set_title(t); ax.set_xlabel("истина"); ax.grid(alpha=0.3)
axs[0].set_ylabel("предсказание")
fig.suptitle(r"$\mathrm{asinh}(R/R_0)$, разбиение «прореживание сетки»")
fig.tight_layout()
fig.savefig(FIG_DIR / "03_pred_vs_true.png", dpi=150, bbox_inches="tight")

## 7. Профили и нулевая поверхность

Проверка того, что модель воспроизводит смену знака в правильном месте.

In [ ]:
T_fix = sorted(core["T"].unique())[0]
sub = core[core["T"] == T_fix].sort_values(["T12", "T3"])
p_all = predict(net, sub)

fig, ax = plt.subplots(figsize=(7, 4.5))
for k, (t12, g) in enumerate(sub.groupby("T12")):
    idx = sub.index.get_indexer(g.index)
    c = plt.cm.viridis(k / 8)
    ax.plot(g["T3"], np.arcsinh(g["R_VV23_12"] / R0), "o", ms=4, color=c)
    ax.plot(g["T3"], np.arcsinh(p_all[idx, 1] / R0), "-", lw=1.2, color=c)
ax.axhline(0, color="k", lw=0.6)
ax.set_xlabel(r"$T_3$, К"); ax.set_ylabel(r"$\mathrm{asinh}(R/R_0)$")
ax.set_title(f"T = {T_fix:g} К — точки: данные, линии: модель")
ax.grid(alpha=0.3)
fig.savefig(FIG_DIR / "03_profiles.png", dpi=150, bbox_inches="tight")

## 8. Скорость

Время одного вызова модели. Это накладные расходы суррогата; прямой расчёт
свёртки поуровневых коэффициентов на порядки дороже.

In [ ]:
print(timing(net, core))
print("\nдля сравнения, сплайн:")
from co2relax.baselines import GridSpline
sp = GridSpline().fit(core, TARGETS[1])
t0 = time.perf_counter()
for _ in range(20):
    sp.predict(core)
print(f"  {(time.perf_counter()-t0)/20/len(core)*1e6:.2f} мкс на точку (один канал)")

## 9. Якобиан

Система уравнений релаксации жёсткая, поэтому в расчёте применяются неявные
схемы, требующие $\partial R/\partial(T, T_{12}, T_3)$. Модель предоставляет
его автоматическим дифференцированием; табличная интерполяция — нет.

In [ ]:
row = te.head(1)
J = jacobian(net, row)
print("форма:", J.shape, "\n")
print(f"{'':16s}{'авто':>14}{'численно':>14}{'отн. разн.':>12}")
for j, t in enumerate(TARGETS):
    for k, c in enumerate(["T", "T12", "T3"]):
        h = 50.0
        p_, m_ = row.copy(), row.copy()
        p_[c] = p_[c] + h; m_[c] = m_[c] - h
        num = (float(predict(net, p_)[0, j]) - float(predict(net, m_)[0, j])) / (2 * h)
        d = abs(J[0, j, k] - num) / max(abs(num), 1e-30)
        print(f"{t[2:-3]:10s} d/d{c:4s}{J[0,j,k]:>14.3e}{num:>14.3e}{d:>12.1e}")

## 10. Устойчивость к инициализации

Разброс по случайным начальным весам — оценка воспроизводимости результата.

In [ ]:
rows = []
for seed in range(5):
    net_s, _ = train_model(tr, Config(model="physics", epochs=1500, seed=seed))
    from co2relax.evaluate import eval_net
    sc = np.array([np.median(np.abs(core[t][core[t] != 0])) for t in TARGETS])
    r = eval_net(net_s, te, sc)
    rows.append({"seed": seed, **{t[2:-3]: r[t]["rel_median"] for t in TARGETS}})
seeds = pd.DataFrame(rows).set_index("seed")
print(seeds)
print("\nразброс (макс/мин):")
print((seeds.max() / seeds.min()).round(2))

## 11. Сводка по устойчивости

Худший случай по всем комбинациям разбиение × канал. Ограничения агрегируются
по максимуму: они либо выполняются везде, либо нарушены.

In [ ]:
summary = res.groupby("модель").agg(
    худшая_медиана=("rel_median", "max"),
    худший_p95=("rel_p95", "max"),
    худшая_невязка=("равновесие", "max"),
    худший_знак=("доля_неверный_знак", "max"),
    минимальный_r2=("r2", "min"),
).sort_values("худшая_медиана")
summary.to_csv(REPORT_DIR / "03_robustness.csv")
summary

## 12. Подбор гиперпараметров

Свип запускается отдельно: `python scripts/sweep.py --seeds 2`.
Здесь только чтение готового результата.

In [ ]:
sw = pd.read_csv(REPORT_DIR / "04_sweep.csv")
agg = (sw.groupby(["hidden", "depth", "epochs"])
         .agg(VT2=("VT2", "mean"), VV23=("VV23", "mean"), VV123=("VV123", "mean"),
              среднее=("среднее", "mean"), эпох=("эпох", "mean"), сек=("сек", "mean"))
         .sort_values("среднее"))

print(f"размах по свипу: {agg['среднее'].max() / agg['среднее'].min():.1f} раза")
print(f"невязка по всем прогонам: {sw['равновесие'].max():.1e}, ошибок знака: {sw['знак'].max():.0f}")
agg.head(8)

## Выводы

**Точность.** После подбора гиперпараметров (4 слоя по 128 нейронов) физически-
информированная сеть даёт худшую по всем комбинациям медианную ошибку 1.0·10⁻²
против 1.3·10⁻² у прямой сплайн-интерполяции и 4.8·10⁻² у сети без физической
структуры. По 95-му процентилю преимущество больше: 4.7·10⁻² против 8.5·10⁻²
у сплайна.

**Физические ограничения.** Модели с факторизацией дают невязку на равновесии
порядка машинного нуля (4.7·10⁻¹⁶) и не допускают ошибок знака. Прямая
интерполяция нарушает равновесие на 2·10⁻², сеть без структуры — на 7.9·10⁻².
Согласно оценке ноутбука 02, невязка такого порядка отвечает паразитному дрейфу
равновесного состояния за время около 10 мкс при газодинамических временах 1 мс.

**Вклад физической структуры.** Сравнение двух сетей одинаковой архитектуры
изолирует эффект факторизации от эффекта самого перехода к нейросетям. Для
канала VT2 выигрыш в точности устойчив и составляет 3.7–8.3 раза; для VV123 —
от 1.1 до 3.8; для VV23 — от 0.7 до 2.6, причём в трёх разбиениях из четырёх
отношение меньше единицы.

Различие объяснимо геометрией: у VT2 нулевая поверхность представляет собой
плоскость T₁₂ = T, содержащую 11% узлов сетки, и её аналитическое знание снимает
с модели существенную часть задачи. У VV-каналов нулевая поверхность двумерна
и занимает меньшую долю области, поэтому выигрыш в точности мал или отсутствует.

Независимо от канала факторизация обеспечивает выполнение физических ограничений:
невязка на равновесии 4.7·10⁻¹⁶ против 7.9·10⁻² у контрольной модели, ошибки
знака отсутствуют. Для VV-каналов ценность структуры состоит именно в этом,
а не в повышении точности.

**Отказ интерполяции множителя.** На экстраполяции по T₃ для канала VV23 модель
«сплайн по A» теряет предсказательную способность полностью: R² = −0.545,
95-й процентиль ошибки 3.0, доля неверного знака 8.1·10⁻². Причина — в
экстраполяции растущей функции A кубическим сплайном, который может выдать
отрицательное значение. Параметризация выхода сети как exp(log A) исключает
такой сценарий конструктивно.

**Скорость.** 1.85 мкс на точку (три канала сразу) против 1.04 мкс у сплайна
на один канал. Сеть не быстрее; обе величины — накладные расходы суррогата,
прямой расчёт свёртки на порядки дороже. Ускорение не является аргументом
в пользу нейросетевого подхода.

**Дифференцируемость.** Якобиан ∂R/∂(T, T₁₂, T₃) вычисляется автоматическим
дифференцированием по всей цепочке, включая множитель; расхождение с центральной
разностью не превышает 0.4%. Табличная интерполяция якобиана не предоставляет,
а жёсткость системы уравнений релаксации требует неявных схем.

**Воспроизводимость.** Разброс медианной ошибки по пяти инициализациям весов
составляет ___ раза (раздел 10). Результат качественно не зависит от случайного
начального приближения.

**Подбор гиперпараметров.** Размах точности по 24 конфигурациям — 6.9 раза;
настроенная конфигурация улучшает базовую в 3.8 раза. При этом по всем 48
прогонам свипа невязка на равновесии не превышает 4.8·10⁻¹⁶, а ошибок знака нет
ни одной: точность настраивается, соблюдение физических ограничений
обеспечивается структурой модели и от конфигурации не зависит.